# 初始 CSV 清理

將 `Data/raw` 的 CSV 清理後輸出至：`Data/process/01_initial_csv_clean`

## 處理內容

- 串流讀取 CSV。
- 保留指定欄位。
- 將未命名第一欄改名為 `row_num`。
- 保留原始文字、時間戳、空值、deleted/removed 留言、重複值與原始編號。
- 輸出格式為 UTF-8 BOM。

## 異常資料

欄位數錯誤的資料不寫入 cleaned CSV，改存至：`.rejected.jsonl`

其中記錄解析後欄位與結束行號。

CSV 語法錯誤會直接停止執行。


In [ ]:
import json
from pathlib import Path
import csv
import sys
import time

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Data" / "raw").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Cannot find Data/raw or input CSV files")

import sys
sys.path.insert(0,str(ROOT/'Notebook'))
from pipeline_io import new_run, publish, input_run, remove_work_database

RAW_DIR = ROOT / "Data" / "raw"
OUTPUT_DIR = new_run(ROOT,'cleaned')
COLUMNS = ["row_num", "author", "author_fullname", "body", "created_utc",
           "id", "link_id", "nest_level", "parent_id"]
PROGRESS_EVERY = 250_000

field_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(field_limit)
        break
    except OverflowError:
        field_limit //= 10

sources = sorted(RAW_DIR.glob("*.csv"))
if not sources:
    raise FileNotFoundError("Cannot find Data/raw or input CSV files")
print(f"Found {len(sources)} files, {sum(p.stat().st_size for p in sources) / 1e9:.2f} GB")


Found 6 files, 2.95 GB


In [2]:
def clean_csv(source, output_dir=OUTPUT_DIR, progress_every=PROGRESS_EVERY):
    source = Path(source)
    output_dir = Path(output_dir)
    destination = output_dir / source.name
    if destination.resolve() == source.resolve():
        raise ValueError("Invalid CSV header or output path")
    output_dir.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    started = time.monotonic()
    count = 0
    rejected = 0
    try:
        with source.open("r", encoding="utf-8-sig", newline="") as incoming:
            reader = csv.reader(incoming, strict=True)
            header = next(reader, None)
            if not header:
                raise ValueError("Invalid CSV header or output path")
            if header[0] in ("", "Unnamed: 0"):
                header[0] = "\u7de8\u865f"
            missing = [name for name in COLUMNS if name not in header]
            if missing:
                raise ValueError("Invalid CSV header or output path")
            if any(header.count(name) != 1 for name in COLUMNS):
                raise ValueError("Invalid CSV header or output path")
            positions = [header.index(name) for name in COLUMNS]
            with temporary.open("w", encoding="utf-8-sig", newline="") as outgoing, destination.with_suffix(".rejected.jsonl").open("w", encoding="utf-8") as rejects:
                writer = csv.writer(outgoing)
                writer.writerow(COLUMNS)
                for row in reader:
                    if len(row) != len(header):
                        rejected += 1
                        rejects.write(json.dumps({"physical_line_end": reader.line_num,
                                                  "fields": row}, ensure_ascii=False) + "\n")
                        continue
                    writer.writerow([row[i] for i in positions])
                    count += 1
                    if progress_every and count % progress_every == 0:
                        print(f"{source.name}: {count:,} rows", flush=True)
        temporary.replace(destination)
        if rejected == 0: destination.with_suffix('.rejected.jsonl').unlink(missing_ok=True)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return {"file": source.name, "rows": count, "rejected_records": rejected,
            "output_bytes": destination.stat().st_size,
            "seconds": round(time.monotonic() - started, 1)}


In [3]:
results = []
for source in sources:
    result = clean_csv(source)
    results.append(result)
    print(result, flush=True)

print(f"Total: {sum(r['rows'] for r in results):,} rows")
print(f"Output: {sum(r['output_bytes'] for r in results) / 1e9:.2f} GB")
print(f"Output directory: {OUTPUT_DIR}")

publish(ROOT,'cleaned',OUTPUT_DIR,{'files':results,'total_rows':sum(r['rows'] for r in results),'input_directory':str(RAW_DIR)})


wallstreetbets_comment_part1.csv: 250,000 rows
{'file': 'wallstreetbets_comment_part1.csv', 'rows': 310753, 'rejected_records': 0, 'output_bytes': 63729671, 'seconds': 1.5}
wallstreetbets_comment_part2.csv: 250,000 rows
wallstreetbets_comment_part2.csv: 500,000 rows
{'file': 'wallstreetbets_comment_part2.csv', 'rows': 575124, 'rejected_records': 24, 'output_bytes': 97837151, 'seconds': 2.5}
wallstreetbets_comment_part3.csv: 250,000 rows
wallstreetbets_comment_part3.csv: 500,000 rows
wallstreetbets_comment_part3.csv: 750,000 rows
wallstreetbets_comment_part3.csv: 1,000,000 rows
wallstreetbets_comment_part3.csv: 1,250,000 rows
wallstreetbets_comment_part3.csv: 1,500,000 rows
wallstreetbets_comment_part3.csv: 1,750,000 rows
wallstreetbets_comment_part3.csv: 2,000,000 rows
{'file': 'wallstreetbets_comment_part3.csv', 'rows': 2218513, 'rejected_records': 0, 'output_bytes': 364124678, 'seconds': 10.4}
wallstreetbets_comment_part4.csv: 250,000 rows
wallstreetbets_comment_part4.csv: 500,000 ro

## 預覽
輸出前三行


In [4]:
from itertools import islice

with (OUTPUT_DIR / sources[0].name).open(encoding="utf-8-sig", newline="") as f:
    preview = list(islice(csv.DictReader(f), 3))
preview


[{'編號': '0',
  'author': 'secondhandsondek',
  'author_fullname': 't2_59t5b',
  'body': 'This is a fantastic idea Ill toss mine up in a bit \n\nJPM and wells fargo this friday BMO possibly facing terrible earningshttpwwwbloombergcomnews20120411profitdropatusbanksimperilsrallyhtml',
  'created_utc': '1334162803.0',
  'id': 'c4b0pvu',
  'link_id': 't3_s4jw1',
  'nest_level': '1',
  'parent_id': 't3_s4jw1'},
 {'編號': '1',
  'author': 'glorysk87',
  'author_fullname': 't2_3zydq',
  'body': 'INTC is on 417',
  'created_utc': '1334165948.0',
  'id': 'c4b1fpf',
  'link_id': 't3_s4jw1',
  'nest_level': '1',
  'parent_id': 't3_s4jw1'},
 {'編號': '2',
  'author': 'entsportsjunkie',
  'author_fullname': 't2_5u44p',
  'body': 'straddle call straddle put put put straddle call straddle call\n\nif i had to pick one for each google i am more bullish but i like straddles on tech ',
  'created_utc': '1334167351.0',
  'id': 'c4b1rmm',
  'link_id': 't3_s4jw1',
  'nest_level': '1',
  'parent_id': 't3_s4jw1'}]